# Restaurant Analytics Agent - Data Foundation

This notebook creates Unity Catalog tables that power a ReAct-pattern agent with four tools:
- **Business Lookup**: Retrieve restaurant ratings, review counts, and metadata
- **Vector Search**: Semantic search over customer reviews (requires review text embeddings)
- **Competitor Comparison**: Compare ratings and trends across restaurants
- **Complaint/Opportunity Extraction**: Identify recurring themes in reviews

**Data pipeline**: Ingest compressed Google Local data → filter San Diego Italian restaurants → create Delta tables in Unity Catalog

**Agent stack**: 
- LLM: Meta Llama 3.3 70B Instruct (Databricks Foundation Model API)
- Embeddings: GTE Large (En) for vector search
- Framework: LangGraph for tool routing

---
## 1. Setup Unity Catalog Structure

Create the catalog, schema, and volume for staging source data and storing agent tables.

**UC namespace**: `main.aai510_final_agent`  
**Volume**: Stores compressed source files for reusable Spark reads (no local `wget` downloads)  
**Source data**:
- `meta-California.json.gz` - Business metadata (name, address, categories, location)
- `rating-California.csv.gz` - User ratings (gmap_id, user_id, rating, timestamp)

In [0]:
from pyspark.sql import functions as F

# Configuration
CATALOG = "main"
SCHEMA = "aai510_final_agent"
VOLUME = "aai510_final_raw"

# Source URLs
META_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/meta-California.json.gz"
RATING_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/rating-California.csv.gz"

# Create Unity Catalog objects
print("=" * 60)
print("Creating Unity Catalog structure...")
print("=" * 60)

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# Volume paths for staged files
META_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/meta-California.json.gz"
RATING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/rating-California.csv.gz"

print(f"\nStaging compressed files to Volume: {CATALOG}.{SCHEMA}.{VOLUME}")
print("This may take a few minutes for large files...\n")

# Stage files into Volume (one-time download, reusable for Spark reads)
dbutils.fs.cp(META_URL, META_PATH)
print(f"✓ Staged: {META_PATH}")

dbutils.fs.cp(RATING_URL, RATING_PATH)
print(f"✓ Staged: {RATING_PATH}")

print(f"\n✓ Unity Catalog setup complete")
print(f"\tCatalog: {CATALOG}")
print(f"\tSchema: {CATALOG}.{SCHEMA}")
print(f"\tVolume: {CATALOG}.{SCHEMA}.{VOLUME}")

---
### 1.5. Inspect Staged Files

Verify that files were successfully staged and examine their schemas to understand the data structure.

**What we're checking**:
- Total record counts for California dataset
- Schema structure (column names and types)
- Sample records to confirm data quality
- Whether review text is available (needed for Vector Search)

This step helps identify any data quality issues early and confirms which agent tools can be implemented with the current dataset.

In [0]:
# Verify staged files and inspect schema
print("=" * 60)
print("Inspecting Staged Files...")
print("=" * 60)

# Read metadata sample
print("\n1. Business Metadata (meta-California.json.gz)")
print("-" * 60)
meta_df = spark.read.json(META_PATH)
print(f"Total businesses in California: {meta_df.count():,}")
print("\nSample records:")
display(meta_df.limit(3))

# Read ratings sample
print("\n2. Ratings Data (rating-California.csv.gz)")
print("-" * 60)
ratings_df = spark.read.option("header", "true").csv(RATING_PATH)
print(f"Total ratings in California: {ratings_df.count():,}")
print("\nSample records:")
display(ratings_df.limit(3))

---
## 2. Filter San Diego Italian Restaurants

Read business metadata and filter for San Diego Italian restaurants using Spark SQL expressions. This creates the foundational `restaurants` table that powers the Business Lookup tool and defines which reviews are relevant for analysis.

**Filters applied**:
- **Geography**: Address contains "San Diego, CA"
- **Category**: Business category includes "Restaurant" AND "Italian"

**Output table**: `main.aai510_final_agent.restaurants`

In [0]:
# Read and filter business metadata
print("=" * 60)
print("Filtering San Diego Italian restaurants...")
print("=" * 60)

# Read metadata with schema inference
meta_df = spark.read.json(META_PATH)

# Filter logic (mimics original regex + category matching)
restaurants_df = (
    meta_df
    # Convert category array to searchable string
    .withColumn("category_text", 
                F.lower(F.concat_ws(" ", F.coalesce(F.col("category"), F.array()))))
    # Normalize address for matching
    .withColumn("address_lower", 
                F.lower(F.coalesce(F.col("address"), F.lit(""))))
    # Apply filters
    .filter(F.col("address_lower").rlike(r",\s*san diego,\s*ca\b"))
    .filter(F.col("category_text").contains("restaurant"))
    .filter(F.col("category_text").contains("italian"))
    # Select clean schema for agent tools
    .select(
        "gmap_id",
        F.col("name").alias("business_name"),
        "address",
        "category",
        "latitude",
        "longitude"
    )
    .dropna(subset=["gmap_id"])
    .dropDuplicates(["gmap_id"])
)

# Preview results
restaurant_count = restaurants_df.count()
print(f"\n✓ Found {restaurant_count:,} San Diego Italian restaurants")

print("\nSample restaurants:")
display(restaurants_df.limit(5))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.restaurants"

print("=" * 60)
print(f"\nWriting to Unity Catalog: {table_name}")
print("=" * 60)

restaurants_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

---
## 3. Join Reviews to Filtered Restaurants

Read the ratings CSV and join to our filtered San Diego Italian restaurants. This creates the `reviews` table that contains all customer ratings for our target restaurants.

**Output table**: `main.aai510_final_agent.reviews`  
**Schema**: `gmap_id`, `user_id`, `rating`, `timestamp`

**Note**: This dataset contains ratings only (no review text). For Vector Search and Complaint Extraction tools, review text will need to be sourced separately.

In [0]:
# Read and join reviews to filtered restaurants
print("=" * 60)
print("Loading and filtering reviews...")
print("=" * 60)

# Read ratings CSV with header
ratings_df = (
    spark.read
    .option("header", "true")
    .csv(RATING_PATH)
    .withColumnRenamed("business", "gmap_id")
    .withColumnRenamed("user", "user_id")
)

print(f"Total ratings in California: {ratings_df.count():,}")

# Load restaurant filter list
restaurants_df = spark.table(f"{CATALOG}.{SCHEMA}.restaurants")
restaurant_ids = restaurants_df.select("gmap_id")

print(f"Filtering to {restaurant_ids.count():,} San Diego Italian restaurants...")

# Inner join to get only reviews for our target restaurants
reviews_df = (
    ratings_df
    .join(restaurant_ids, on="gmap_id", how="inner")
    .select("gmap_id", "user_id", "rating", "timestamp")
)

review_count = reviews_df.count()
print(f"\n✓ Found {review_count:,} reviews for San Diego Italian restaurants")

print("\nSample reviews:")
display(reviews_df.limit(5))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.reviews"
print(f"\nWriting to Unity Catalog: {table_name}")
reviews_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

# Summary stats
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Restaurants: {restaurant_ids.count():,}")
print(f"Reviews: {review_count:,}")
print(f"Avg reviews per restaurant: {review_count / restaurant_ids.count():.1f}")

---
## 4. Create Restaurant Metrics Table

Aggregate review data to create metrics for each restaurant. This table powers the **Business Lookup** and **Competitor Comparison** agent tools.

**Output table**: `main.aai510_final_agent.restaurant_metrics`  
**Metrics computed**:
- Average rating
- Total review count
- Rating distribution (count by 1-5 stars)
- Latest review timestamp
- Restaurant metadata (name, address, location)

In [0]:
# Create aggregated metrics from reviews
print("=" * 60)
print("Computing restaurant metrics...")
print("=" * 60)

# Load reviews and restaurants
reviews_df = spark.table(f"{CATALOG}.{SCHEMA}.reviews")
restaurants_df = spark.table(f"{CATALOG}.{SCHEMA}.restaurants")

# Aggregate metrics per restaurant
metrics_df = (
    reviews_df
    .groupBy("gmap_id")
    .agg(
        F.avg("rating").alias("avg_rating"),
        F.count("*").alias("review_count"),
        F.max("timestamp").alias("latest_review_timestamp"),
        # Rating distribution: count by rating value (1-5 stars)
        F.sum(F.when(F.col("rating") == "1", 1).otherwise(0)).alias("rating_1_count"),
        F.sum(F.when(F.col("rating") == "2", 1).otherwise(0)).alias("rating_2_count"),
        F.sum(F.when(F.col("rating") == "3", 1).otherwise(0)).alias("rating_3_count"),
        F.sum(F.when(F.col("rating") == "4", 1).otherwise(0)).alias("rating_4_count"),
        F.sum(F.when(F.col("rating") == "5", 1).otherwise(0)).alias("rating_5_count"),
    )
)

# Join with restaurant metadata
restaurant_metrics_df = (
    metrics_df
    .join(restaurants_df, on="gmap_id", how="inner")
    .select(
        "gmap_id",
        "business_name",
        "address",
        "category",
        "latitude",
        "longitude",
        "avg_rating",
        "review_count",
        "latest_review_timestamp",
        "rating_1_count",
        "rating_2_count",
        "rating_3_count",
        "rating_4_count",
        "rating_5_count",
    )
    .orderBy(F.desc("review_count"))
)

print(f"\n✓ Computed metrics for {restaurant_metrics_df.count():,} restaurants")

print("\nTop 10 restaurants by review count:")
display(restaurant_metrics_df.limit(10))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
print(f"\nWriting to Unity Catalog: {table_name}")
restaurant_metrics_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

print("\n" + "=" * 60)
print("✓ Restaurant metrics table ready for agent tools")
print("=" * 60)

---
## 5. Prepare for Vector Search (Review Text)

The current `reviews` table only contains ratings (1-5 stars) without review text. For **Vector Search** and **Complaint/Opportunity Extraction** tools, we need the actual review text to create embeddings.

**Requirements for Vector Search**:
- Review text field for GTE Large embeddings
- Join reviews to filtered San Diego Italian restaurants
- Create Databricks Vector Search index

**Data source**: `review-California.json.gz` contains full review data including text field

In [ ]:
# Prepare Review Text for Theme Retrieval

from pyspark.sql import functions as F

# Review text configuration
REVIEW_TEXT_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/review-California.json.gz"
REVIEW_TEXT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/review-California.json.gz"
REVIEW_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"

print("=" * 60)
print("Preparing review text for Theme Retrieval...")
print("=" * 60)

print(f"\nReview text source: {REVIEW_TEXT_URL}")
print(f"Review text path: {REVIEW_TEXT_PATH}")
print(f"Review text table: {REVIEW_TEXT_TABLE}")

In [ ]:
# Check whether review text table already exists
try:
    reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)
    
    print(f"✓ Found existing table: {REVIEW_TEXT_TABLE}")
    print(f"✓ Total review-text rows: {reviews_with_text_df.count():,}")
    
    display(reviews_with_text_df.limit(5))
    
    REVIEW_TEXT_READY = True

except Exception:
    print(f"Review text table not found: {REVIEW_TEXT_TABLE}")
    print("Checking whether raw review text file is already staged...")
    
    REVIEW_TEXT_READY = False

In [ ]:
# Stage raw review text file only if needed
if not REVIEW_TEXT_READY:
    
    try:
        dbutils.fs.ls(REVIEW_TEXT_PATH)
        print(f"✓ Already staged: {REVIEW_TEXT_PATH}")
        
    except Exception:
        print("Raw review text file not found.")
        print("Staging raw review text file. This may take several minutes...")
        
        dbutils.fs.cp(REVIEW_TEXT_URL, REVIEW_TEXT_PATH)
        
        print(f"✓ Staged: {REVIEW_TEXT_PATH}")

else:
    print("Skipping raw file staging because review text table already exists.")

In [ ]:
# Create review text table only if needed
if not REVIEW_TEXT_READY:
    
    print("=" * 60)
    print("Creating reviews_with_text table...")
    print("=" * 60)
    
    # Read raw review text file
    reviews_with_text_raw_df = spark.read.json(REVIEW_TEXT_PATH)
    
    # Get filtered restaurant IDs from Section 2
    restaurant_ids_df = (
        spark.table(f"{CATALOG}.{SCHEMA}.restaurants")
        .select("gmap_id")
        .distinct()
    )
    
    # Keep only written reviews for selected restaurants
    reviews_with_text_filtered_df = (
        reviews_with_text_raw_df
        .join(restaurant_ids_df, on="gmap_id", how="inner")
        .select(
            "gmap_id",
            "user_id",
            "rating",
            "time",
            "text",
            "name"
        )
        .filter(F.col("text").isNotNull())
        .filter(F.length(F.col("text")) > 10)
    )
    
    # Save filtered review text table
    (
        reviews_with_text_filtered_df
        .write
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(REVIEW_TEXT_TABLE)
    )
    
    reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)
    
    print(f"✓ Created table: {REVIEW_TEXT_TABLE}")
    print(f"✓ Total review-text rows: {reviews_with_text_df.count():,}")
    
    display(reviews_with_text_df.limit(5))

else:
    print("Skipping table creation because review text table already exists.")

In [ ]:
# Verify review text table
print("=" * 60)
print("Verifying review text table...")
print("=" * 60)

reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)

print(f"Table: {REVIEW_TEXT_TABLE}")
print(f"Total rows: {reviews_with_text_df.count():,}")

display(
    reviews_with_text_df
    .select("gmap_id", "rating", "text")
    .limit(10)
)

---
## 6. Validation Queries - Demonstrate Agent Tool Patterns

Test queries that simulate how each agent tool will interact with the Unity Catalog tables. These patterns will be implemented as tool functions in the LangGraph agent.

**Tools validated**:
1. **Business Lookup**: Retrieve metrics for a specific restaurant
2. **Competitor Comparison**: Compare restaurant performance
3. **Review Retrieval**: Sample reviews for analysis (placeholder for Vector Search)
4. **Trend Analysis**: Rating distribution and temporal patterns

In [0]:
# Load tables for validation
restaurant_metrics = spark.table(f"{CATALOG}.{SCHEMA}.restaurant_metrics")
reviews = spark.table(f"{CATALOG}.{SCHEMA}.reviews")

# ============================================================
# Tool 1: Business Lookup
# ============================================================
print("\nTOOL 1: Business Lookup")
print("-" * 60)
print("Query: 'What is my restaurant's average rating and review count?'\n")

# Example: Get metrics for top restaurant
sample_restaurant = restaurant_metrics.first()
gmap_id_example = sample_restaurant["gmap_id"]

business_lookup_result = restaurant_metrics.filter(F.col("gmap_id") == gmap_id_example)
display(business_lookup_result)

print(f"\nAgent response would include:")
print(f"  • Restaurant: {sample_restaurant['business_name']}")
print(f"  • Average rating: {sample_restaurant['avg_rating']:.2f} stars")
print(f"  • Total reviews: {sample_restaurant['review_count']:,}")
print(f"  • Rating breakdown: 5★({sample_restaurant['rating_5_count']}) 4★({sample_restaurant['rating_4_count']}) 3★({sample_restaurant['rating_3_count']}) 2★({sample_restaurant['rating_2_count']}) 1★({sample_restaurant['rating_1_count']})")

In [0]:
# ============================================================
# Tool 2: Competitor Comparison
# ============================================================
print("\nTOOL 2: Competitor Comparison")
print("-" * 60)
print("Query: 'How do I compare to my top competitors?'\n")

top_competitors = restaurant_metrics.orderBy(F.desc("review_count")).limit(5)
display(top_competitors.select("business_name", "avg_rating", "review_count", "address"))

print("\nAgent response would analyze:")
print("  • Ranking by review volume")
print("  • Average rating comparison")
print("  • Geographic proximity")

In [0]:
# ============================================================
# Tool 3: Review Retrieval (Placeholder for Vector Search)
# ============================================================
print("\nTOOL 3: Review Retrieval")
print("-" * 60)
print("Query: 'Show me recent reviews for restaurant X'\n")

sample_reviews = (
    reviews
    .filter(F.col("gmap_id") == gmap_id_example)
    .orderBy(F.desc("timestamp"))
    .limit(5)
)
display(sample_reviews)

print("\n⚠️  NOTE: Current dataset has ratings only (no review text)")
print("   • For Vector Search: Need review-California.json.gz with text field")
print("   • For Complaint Extraction: Need review text for LLM analysis")
print("   • Current capability: Can filter by rating, timestamp, restaurant")

In [0]:
# ============================================================
# Tool 4: Trend Analysis
# ============================================================
print("\nTOOL 4: Trend Analysis")
print("-" * 60)
print("Query: 'What are the rating trends across all restaurants?'\n")

rating_distribution_summary = restaurant_metrics.agg(
    F.avg("avg_rating").alias("overall_avg_rating"),
    F.sum("review_count").alias("total_reviews"),
    F.sum("rating_5_count").alias("total_5_star"),
    F.sum("rating_4_count").alias("total_4_star"),
    F.sum("rating_3_count").alias("total_3_star"),
    F.sum("rating_2_count").alias("total_2_star"),
    F.sum("rating_1_count").alias("total_1_star"),
)
display(rating_distribution_summary)
